In [ ]:
def get_anchor_id(mol, signal):
    for atom in mol.GetAtoms():
        if atom.GetSymbol()==signal:
            neighbors=atom.GetNeighbors()
            return neighbors[0].GetIdx()
    return None

def get_signal_id(mol, signal):
    for atom in mol.GetAtoms():
        if atom.GetSymbol()==signal:
            return atom.GetIdx()
    return None

def combine2frags(mol1,mol2,signal1,signal2):
    merged_mol=Chem.CombineMols(mol1,mol2)
    anchor1=get_anchor_id(merged_mol,signal1)
    anchor2=get_anchor_id(merged_mol,signal2)

    # link the fragments
    edi_mol=Chem.EditableMol(merged_mol)
    edi_mol.AddBond(anchor1,anchor2,order=Chem.rdchem.BondType.SINGLE)
    # remove the signals
    signal1=get_signal_id(merged_mol,signal1)
    edi_mol.RemoveAtom(signal1)
    new_mol=edi_mol.GetMol()
    signal2=get_signal_id(new_mol,signal2)
    edi_mol=Chem.EditableMol(new_mol)
    edi_mol.RemoveAtom(signal2)
    final_mol=edi_mol.GetMol()
    return final_mol

In [ ]:
from rdkit import Chem
with open('generated_linkers.csv') as f:
    linkers = [line.strip('\n').split(',') for line in f]
with open('data/PROTAC.csv') as f: # change the path to your generated molecules
    data = [line.strip('\n').split(',') for line in f][1:]
full = []
for p in linkers:
    inf = data[int(p[0])]
    frag1 = Chem.MolFromSmiles(inf[-3].replace('*','Fr'))
    frag2 = Chem.MolFromSmiles(inf[-1].replace('*','Rb'))
    linker = Chem.MolFromSmiles(p[1].replace('*','Cs',1).replace('*','K'))
    combine1=combine2frags(linker,frag1,'Fr','Cs')
    combine2=combine2frags(combine1,frag2,'K','Rb')
    if Chem.MolFromSmiles(Chem.MolToSmiles(combine2)):
        full.append(combine2)
    
print('All mols: ', len(full))

In [ ]:
uniq = []
for mol in full:
    uniq_smi = Chem.MolToSmiles(mol) # standardize
    if uniq_smi not in uniq:
        uniq.append(uniq_smi)
print('Uniqueness: ', len(uniq)/len(full))

Uniqueness:  0.104


In [ ]:
# save the complete molecules
with open('complete_molecules.csv','w') as f:
    for mol in full:
        f.write(Chem.MolToSmiles(mol)+'\n')